# Building AI Agents

Welcome to Module 7! In this notebook, we will explore the concept of AI Agents and how they differ from simple language models.

## What We Will Learn

An AI Agent follows this powerful flow:

**User Input → LLM Decision-Making → Tool/Action → Tool Result → LLM Processing → Final Response**

This is fundamentally different from a simple LLM that just generates a response. Agents enable applications to:
- Think about what action is needed
- Call external tools or functions
- Process the results
- Continue working toward a goal

By the end of this module, you will understand how to build intelligent applications that can reason, decide, and act.

## LLM vs. Agent: The Key Difference

### A Normal LLM Application

```
User Input → LLM → Response
```

The LLM reads the user's request and generates an answer. It has no way to interact with external systems.

### An Agentic Application

```
User Input → LLM → Decision → Action/Tool → Result → LLM → Final Response
```

The LLM acts as a **decision-maker**. Instead of directly answering, it:
1. Understands the user's request
2. Decides which action (tool) is needed
3. Returns the decision to your application
4. Your application executes the tool
5. The tool's result is sent back to the LLM
6. The LLM generates the final response

### The Key Shift

**An LLM generates an answer, while an agent uses an LLM to decide what action should happen next.**

## Four Key Concepts

Let's clarify the difference between these related but distinct ideas:

| Concept | Definition |
|---------|------------|
| **LLM** | A neural network that understands and generates text based on patterns in training data. |
| **RAG** | Retrieval-Augmented Generation. Gives the LLM access to external knowledge by retrieving relevant documents and including them in the prompt. |
| **Tool** | A function or capability that the application exposes to allow real-world interactions (e.g., calculating, searching a database, sending an email). |
| **Agent** | Uses an LLM to dynamically decide which tools to call and in what order to achieve a goal. The LLM reason about the task and directs the application's actions. |

**Visual Summary:**
- LLM = Understands/generates language
- RAG = Gives an LLM external knowledge
- Tool = Gives the application a capability
- Agent = Uses an LLM to dynamically decide actions/tools and continue execution

## Why Do We Need Agents?

An LLM is powerful at understanding language, but it **cannot magically access external systems**.

### Real-World Tasks Require Tools

Consider these scenarios:

1. **User asks:** "What is 25 × 48?"
   - The LLM might generate "1200" from training data, but it doesn't *calculate*.
   - A **calculator tool** gives the application the ability to compute this accurately.

2. **User asks:** "Check if customer C001 exists."
   - The LLM has no access to your database.
   - A **database lookup tool** connects the agent to your customer data.

3. **User asks:** "Search my documents for refund policies."
   - The LLM doesn't have access to your files.
   - A **document search tool** retrieves the information.

4. **User asks:** "Send a notification to the customer."
   - The LLM cannot send emails or messages.
   - An **email/notification tool** executes the action.

### The Solution

**Tools give the application capabilities.** By combining the LLM's reasoning with tools, we build intelligent agents that can interact with the real world.

## Building Blocks: From LLM to Agent

Before we build a full agent, we start with a simpler concept: **Using the LLM as a decision-maker or router.**

Imagine a customer support system. When a customer's request comes in, the LLM needs to decide:
- Is this a **calculator** request?
- Is this a **weather** request?
- Is this a **knowledge/FAQ** request?

The LLM acts as a **router** that categorizes the user's intent.

### Example Capabilities

- `calculator` → Perform mathematical operations
- `weather` → Fetch weather information
- `knowledge` → Answer general questions

Once the LLM decides which capability is needed, your application can take the appropriate action.

This is the foundation of agent reasoning!

In [1]:
# Import OpenAI and load the API key from .env
from openai import OpenAI
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Initialize the OpenAI client
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✓ OpenAI client initialized successfully!")

✓ OpenAI client initialized successfully!


In [2]:
# Simple Routing Example: LLM as a Decision-Maker

# Define the system prompt that instructs the LLM to act as a router
system_prompt = """
You are a routing agent. Your job is to categorize user requests into ONE of these categories:
- calculator: mathematical calculations
- weather: weather-related queries
- knowledge: general knowledge or FAQ questions

Respond with ONLY the category name (no explanation, no punctuation).
"""

# User query
user_query = "How much is 25 multiplied by 48?"

# Call the LLM
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ],
    temperature=0  # Deterministic output for routing
)

# Extract the decision
decision = response.choices[0].message.content.strip()

print(f"User Query: {user_query}")
print(f"LLM Decision: {decision}")

User Query: How much is 25 multiplied by 48?
LLM Decision: calculator


## Key Insight: Decision ≠ Action

**Important:** When the LLM returns `calculator`, Python has **NOT** performed the calculation yet.

The LLM only **decided** which capability is needed.

### The Flow

```
User Query: "How much is 25 × 48?"
   ↓
LLM Decision: "calculator"
   ↓
Python Application: "Okay, I need to call the calculator tool"
   ↓
calculator(25, 48) → 1200
```

The separation of concerns is crucial:
- **LLM** → understands intent and makes decisions
- **Application** → executes actions based on decisions

In [3]:
# Multiple Routing Examples

test_queries = [
    "Should I carry an umbrella in Chennai today?",
    "Explain the difference between RAG and fine-tuning.",
    "What is 100 divided by 5?"
]

system_prompt = """
You are a routing agent. Your job is to categorize user requests into ONE of these categories:
- calculator: mathematical calculations
- weather: weather-related queries
- knowledge: general knowledge or FAQ questions

Respond with ONLY the category name (no explanation, no punctuation).
"""

print("Routing Examples:\n")
for query in test_queries:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0
    )
    decision = response.choices[0].message.content.strip()
    print(f"Query: {query}")
    print(f"Decision: {decision}\n")

Routing Examples:

Query: Should I carry an umbrella in Chennai today?
Decision: weather

Query: Explain the difference between RAG and fine-tuning.
Decision: knowledge

Query: What is 100 divided by 5?
Decision: calculator



## Structured Outputs: Making Decisions Machine-Readable

Simple text responses like "calculator" work, but **structured outputs** are much better for real applications.

### Why Structured Outputs?

**Instead of this:**
```
calculator
```

**Use this:**
```json
{
  "action": "calculator",
  "operation": "multiply",
  "a": 25,
  "b": 48
}
```

**Benefits:**
- Easy for Python to parse and extract data
- Includes important details (operation, operands)
- Reduces parsing errors
- Makes the agent's reasoning transparent

Structured outputs make model decisions easier for Python applications to consume and act upon.

## Introduction to Tools

Now we move from simple routing to **actual tool calling**.

### The Tool Calling Flow

```
User Request
   ↓
LLM analyzes and decides: "I need to call the calculator tool"
   ↓
LLM provides: function name, arguments
   ↓
Application executes the Python function
   ↓
Tool returns result
   ↓
LLM processes result
   ↓
Final Response to User
```

### What is a Tool?

A **tool** is simply a function or capability that we expose to the LLM. The tool:
- Has a **name** (e.g., `calculator`)
- Has a **description** (e.g., "Perform arithmetic operations")
- Has **parameters** (e.g., `a`, `b`, `operation`)
- Has a **Python implementation**

The LLM uses the tool definition to decide when and how to use it.

## Built-In Tools vs. Custom Tools

### Built-In Tools

Built-in tools are provided by the AI platform (e.g., OpenAI):
- **Web Search** → Search the internet
- **File/Document Search** → Query document repositories
- **Code Execution** → Run code in a sandbox
- **Calendar Integration** → Access scheduling information

These are pre-built and managed by the platform.

### Custom Tools

Custom tools are **functions we write ourselves** to expose capabilities specific to our application:
- `calculator()` → Perform calculations
- `get_weather()` → Fetch weather data
- `get_customer()` → Query customer database
- `search_products()` → Search product inventory
- `process_refund()` → Execute a refund

### In This Course

We focus on **custom tools** because they demonstrate the core agent concepts. The principles are the same whether using built-in or custom tools.

## Creating Custom Tools

A custom tool is just a **Python function** that we define and make available to the LLM.

### Simple Examples

```python
def calculator(a, b, operation):
    """Perform basic arithmetic operations."""
    if operation == "add":
        return a + b
    elif operation == "multiply":
        return a * b

def get_customer(customer_id):
    """Retrieve customer information from database."""
    # Database lookup logic here
    return customer_data

def search_products(query):
    """Search for products by name or category."""
    # Search logic here
    return results
```

### Key Point

We **define** these functions in Python, then tell the LLM they exist. The LLM doesn't execute the code—it just decides when to call them.

In [4]:
# Define a Simple Custom Tool

def simple_calculator(a, b):
    """
    A very simple tool that adds two numbers.
    This demonstrates the concept before we add complexity.
    """
    result = a + b
    return result

# Test the function
result = simple_calculator(10, 20)
print(f"simple_calculator(10, 20) = {result}")
print("\n✓ Simple custom tool defined and working!")

simple_calculator(10, 20) = 30

✓ Simple custom tool defined and working!


## The Tool Execution Cycle

It's critical to understand that **the LLM does not execute Python directly.**

### The Flow

```
Step 1: User asks "What is 25 + 48?"
   ↓
Step 2: LLM analyzes the request
   ↓
Step 3: LLM decides "I should call the calculator tool"
   ↓
Step 4: LLM returns to the application:
        "Call calculator with a=25, b=48"
   ↓
Step 5: Application executes Python function
        calculator(25, 48) → 73
   ↓
Step 6: Tool result (73) is sent back to LLM
   ↓
Step 7: LLM generates the final response
        "The answer is 73."
```

### Key Takeaway

Your application is the **orchestrator**. The LLM makes decisions, but your code executes them.

In [5]:
# Tool Calling with OpenAI: Add Two Numbers

# Step 1: Define the Python function
def add_numbers(a, b):
    """Add two numbers together."""
    return a + b

# Step 2: Define the tool in a format the LLM understands
tools = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Add two numbers together",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "number",
                        "description": "First number"
                    },
                    "b": {
                        "type": "number",
                        "description": "Second number"
                    }
                },
                "required": ["a", "b"]
            }
        }
    }
]

# Step 3: User query
user_query = "What is 25 plus 48?"

# Step 4: Call the LLM with tools
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": user_query}],
    tools=tools,
    tool_choice="auto"  # Let the LLM decide whether to use a tool
)

# Step 5: Check if the LLM decided to call a tool
assistant_message = response.choices[0].message
print(f"User Query: {user_query}\n")

if assistant_message.tool_calls:
    # The LLM decided to call a tool
    tool_call = assistant_message.tool_calls[0]
    print(f"LLM Decision: Call {tool_call.function.name}")
    print(f"Arguments: {tool_call.function.arguments}")
    
    # Step 6: Execute the Python function
    import json
    args = json.loads(tool_call.function.arguments)
    tool_result = add_numbers(args['a'], args['b'])
    print(f"Tool Result: {tool_result}")
    
    # Step 7: Send the result back to the LLM
    response2 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": user_query},
            assistant_message,
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(tool_result)
            }
        ],
        tools=tools
    )
    
    # Step 8: Get the final response
    final_response = response2.choices[0].message.content
    print(f"\nFinal Response: {final_response}")
else:
    print("LLM did not call a tool.")

User Query: What is 25 plus 48?

LLM Decision: Call add_numbers
Arguments: {"a":25,"b":48}
Tool Result: 73

Final Response: 25 plus 48 equals 73.


## Anatomy of a Tool Definition

When you expose a tool to the LLM, you provide a specification that tells the model what the tool does and how to use it.

### Four Essential Parts

```json
{
  "name": "add_numbers",
  "description": "Add two numbers together",
  "parameters": {
    "type": "object",
    "properties": {
      "a": {"type": "number", "description": "First number"},
      "b": {"type": "number", "description": "Second number"}
    },
    "required": ["a", "b"]
  }
}
```

| Part | Purpose |
|------|----------|
| **name** | The identifier of the tool (e.g., `add_numbers`, `get_customer`) |
| **description** | A clear explanation of what the tool does. The LLM uses this to decide when to use it. |
| **parameters** | The inputs the tool expects (types, descriptions, required fields) |
| **Python implementation** | The actual function that executes when called |

### Why Description Matters

**Good description:** "Retrieve complete customer information including name, email, phone, and account status."

**Bad description:** "Get customer."

The LLM uses the description to decide when and how to use the tool. Clear descriptions lead to better decisions.

In [ ]:
# Custom Tool Example: Get Customer Information

# Step 1: Create an in-memory customer database
customer_database = {
    "C001": {
        "name": "Raj Kumar",
        "email": "raj@example.com",
        "phone": "+91-9876543210",
        "status": "active"
    },
    "C002": {
        "name": "Priya Sharma",
        "email": "priya@example.com",
        "phone": "+91-9123456789",
        "status": "active"
    }
}

# Step 2: Define the Python function
def get_customer(customer_id):
    """Retrieve customer information from the database."""
    if customer_id in customer_database:
        return customer_database[customer_id]
    else:
        return {"error": f"Customer {customer_id} not found"}

# Step 3: Define the tool for the LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_customer",
            "description": "Retrieve complete customer information including name, email, phone number, and account status.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {
                        "type": "string",
                        "description": "The unique customer identifier (e.g., C001, C002)"
                    }
                },
                "required": ["customer_id"]
            }
        }
    }
]

# Step 4: User query
user_query = "Get the details of customer C001."

# Step 5: Call the LLM
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": user_query}],
    tools=tools,
    tool_choice="auto"
)

# Step 6: Process the LLM's decision
import json
assistant_message = response.choices[0].message
print(f"User Query: {user_query}\n")

if assistant_message.tool_calls:
    tool_call = assistant_message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    
    print(f"LLM Decision: Call {tool_call.function.name}")
    print(f"Arguments: customer_id = {args['customer_id']}")
    
    # Step 7: Execute the Python function
    tool_result = get_customer(args['customer_id'])
    print(f"\nTool Result:")
    print(json.dumps(tool_result, indent=2))
    
    # Step 8: Send the result back to LLM
    response2 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": user_query},
            assistant_message,
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result)
            }
        ],
        tools=tools
    )
    
    final_response = response2.choices[0].message.content
    print(f"\nFinal Response:\n{final_response}")

## The Complete Agent Loop

Now we understand the full cycle of an agentic system.

### Agent Loop with Tools

```
                    ┌─────────────────┐
                    │   User Input    │
                    └────────┬────────┘
                             │
                             ▼
                    ┌─────────────────┐
                    │      LLM        │
                    │   (with tools)  │
                    └────────┬────────┘
                             │
                    ┌────────▼────────┐
                    │ Does it need    │
                    │ a tool?         │
                    └────┬────────┬───┘
                         │        │
                        NO       YES
                         │        │
                         │        ▼
                         │   ┌─────────────┐
                         │   │ Tool Call   │
                         │   │ (function)  │
                         │   └──────┬──────┘
                         │          │
                         │          ▼
                         │   ┌─────────────┐
                         │   │ Tool Result │
                         │   └──────┬──────┘
                         │          │
                         │          ▼
                         │   ┌─────────────┐
                         │   │  LLM (2nd   │
                         │   │  round)     │
                         │   └──────┬──────┘
                         │          │
                         └────┬─────┘
                              │
                              ▼
                    ┌─────────────────┐
                    │ Final Response  │
                    └─────────────────┘
```

### Two Paths

1. **No Tool Needed:** LLM → Final Response
2. **Tool Needed:** LLM → Tool Call → Tool Result → LLM → Final Response

In [6]:
# Generic Agent Loop

import json
from typing import Any

# Define two simple tools
def calculator(a, b, operation):
    """Perform arithmetic operations."""
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        return a / b if b != 0 else "Error: Division by zero"
    return "Unknown operation"

customer_db = {
    "C001": {"name": "Raj Kumar", "account_balance": 5000},
    "C002": {"name": "Priya Sharma", "account_balance": 10000}
}

def get_customer(customer_id):
    """Get customer information."""
    return customer_db.get(customer_id, {"error": "Customer not found"})

# Map function names to actual Python functions
function_map = {
    "calculator": calculator,
    "get_customer": get_customer
}

# Define tools for the LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Perform arithmetic operations (add, subtract, multiply, divide)",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"},
                    "operation": {"type": "string", "enum": ["add", "subtract", "multiply", "divide"]}
                },
                "required": ["a", "b", "operation"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_customer",
            "description": "Retrieve customer information by ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string"}
                },
                "required": ["customer_id"]
            }
        }
    }
]

# Agent loop function
def run_agent(user_query: str) -> str:
    """Run the agent loop until we get a final response."""
    print(f"User: {user_query}\n")
    
    messages = [{"role": "user", "content": user_query}]
    
    # Loop until we get a final response (not a tool call)
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        assistant_message = response.choices[0].message
        messages.append(assistant_message)
        
        # Check if LLM wants to call a tool
        if not assistant_message.tool_calls:
            # No tool call - we have the final response
            return assistant_message.content
        
        # Process each tool call
        for tool_call in assistant_message.tool_calls:
            print(f"Tool Called: {tool_call.function.name}")
            
            # Parse arguments
            args = json.loads(tool_call.function.arguments)
            print(f"Arguments: {args}")
            
            # Execute the tool
            func = function_map[tool_call.function.name]
            result = func(**args)
            print(f"Result: {result}\n")
            
            # Add tool result to messages
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

# Test the agent
final_response = run_agent("What is 100 multiplied by 3?")
print(f"Assistant: {final_response}")

User: What is 100 multiplied by 3?

Tool Called: calculator
Arguments: {'a': 100, 'b': 3, 'operation': 'multiply'}
Result: 300

Assistant: 100 multiplied by 3 is 300.


## Task-Oriented Agents

So far, we've seen agents that call a single tool. A **task-oriented agent** is more powerful—it can:

1. **Understand the goal** → Parse the user's request
2. **Choose tools strategically** → Decide which tool to call first
3. **Inspect results** → Analyze what the tool returned
4. **Decide the next step** → Determine if more tools are needed
5. **Continue execution** → Repeat until the task is complete

### Example: Multi-Step Task

**User Request:** "Find customer C001 and calculate a 10% discount on their account balance."

**Agent Flow:**
```
Step 1: Call get_customer("C001")
        → Result: {"name": "Raj Kumar", "account_balance": 5000}

Step 2: Call calculator(5000, 0.10, "multiply")
        → Result: 500

Step 3: LLM synthesizes: "The 10% discount is ₹500."
```

The agent **chains** multiple tool calls to accomplish a complex task.

### Key Characteristic

The LLM decides not just *which* tool to use, but also *when* and *in what order*.

In [7]:
# Task-Oriented Agent Example: Get Customer and Calculate Discount

import json

# Tools
def get_customer(customer_id):
    """Get customer information."""
    customers = {
        "C001": {"name": "Raj Kumar", "account_balance": 5000},
        "C002": {"name": "Priya Sharma", "account_balance": 10000}
    }
    return customers.get(customer_id, {"error": "Not found"})

def calculate_discount(amount, discount_percent):
    """Calculate discount amount."""
    return amount * (discount_percent / 100)

# Function mapping
functions = {
    "get_customer": get_customer,
    "calculate_discount": calculate_discount
}

# Tool definitions
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_customer",
            "description": "Get customer information including name and account balance",
            "parameters": {
                "type": "object",
                "properties": {"customer_id": {"type": "string"}},
                "required": ["customer_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_discount",
            "description": "Calculate discount amount based on original amount and discount percentage",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number"},
                    "discount_percent": {"type": "number"}
                },
                "required": ["amount", "discount_percent"]
            }
        }
    }
]

# Agent loop
def task_oriented_agent(user_request):
    print(f"User Request: {user_request}\n")
    
    messages = [{"role": "user", "content": user_request}]
    
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        assistant_message = response.choices[0].message
        messages.append(assistant_message)
        
        if not assistant_message.tool_calls:
            return assistant_message.content
        
        for tool_call in assistant_message.tool_calls:
            print(f"→ Calling {tool_call.function.name}")
            args = json.loads(tool_call.function.arguments)
            print(f"  Arguments: {args}")
            
            result = functions[tool_call.function.name](**args)
            print(f"  Result: {result}\n")
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

# Test multi-step task
response = task_oriented_agent("Find customer C001 and calculate a 10% discount on their account balance.")
print(f"Final Response:\n{response}")

User Request: Find customer C001 and calculate a 10% discount on their account balance.

→ Calling get_customer
  Arguments: {'customer_id': 'C001'}
  Result: {'name': 'Raj Kumar', 'account_balance': 5000}

→ Calling calculate_discount
  Arguments: {'amount': 5000, 'discount_percent': 10}
  Result: 500.0

Final Response:
The customer C001, Raj Kumar, has an account balance of $5000. A 10% discount on this amount is $500.


## Guardrails: Protecting Critical Operations

An important lesson: **Never rely on the LLM alone to enforce critical business rules.**

### Real-World Scenario: Refund Processing

Imagine a customer support system:

**Customer Request:** "My order (Order 1001) arrived damaged. Please refund me."

**Order Details:**
- Order ID: 1001
- Amount: ₹1800
- Condition: Damaged (eligible for refund)

**Company Policy:**
- Refunds ≤ ₹1000 → **Automatic processing**
- Refunds > ₹1000 → **Require human approval**

### The Problem

If we let the LLM handle the entire refund decision:
- The LLM might misunderstand the policy
- The LLM could be manipulated
- There's no audit trail

### The Correct Architecture

```
LLM Decision → Python Guardrail → Human Approval → Tool Execution
```

**Flow:**
1. **LLM analyzes:** "This is a refund request for ₹1800."
2. **Python guardrail:** "₹1800 > ₹1000, so human approval is required."
3. **Human reviews:** Manager approves/denies the refund.
4. **Tool executes:** If approved, `process_refund()` is called.

### Key Principle

**Deterministic business rules should stay in application code, not in the LLM.**

In [8]:
# Simplified Refund Guardrail Example

def process_refund_with_guardrail(customer_id, order_id, refund_amount):
    """
    Process a refund with business logic guardrails.
    The LLM suggests the refund, but Python enforces the rules.
    """
    
    # Company policy thresholds
    AUTO_REFUND_LIMIT = 1000  # ₹1000
    
    print(f"Processing Refund:")
    print(f"  Customer ID: {customer_id}")
    print(f"  Order ID: {order_id}")
    print(f"  Refund Amount: ₹{refund_amount}")
    print()
    
    # Guardrail 1: Check refund amount
    if refund_amount <= 0:
        return {"status": "REJECTED", "reason": "Invalid refund amount"}
    
    # Guardrail 2: Apply business logic
    if refund_amount <= AUTO_REFUND_LIMIT:
        # Automatic approval
        print(f"✓ Guardrail Check: Refund ≤ ₹{AUTO_REFUND_LIMIT}")
        print(f"✓ Decision: AUTOMATIC APPROVAL")
        print(f"✓ Action: Processing refund...")
        return {"status": "APPROVED", "approval_type": "automatic", "amount": refund_amount}
    else:
        # Requires human approval
        print(f"⚠ Guardrail Check: Refund > ₹{AUTO_REFUND_LIMIT}")
        print(f"⚠ Decision: HUMAN APPROVAL REQUIRED")
        print(f"⚠ Action: Forwarding to manager for review...")
        return {"status": "PENDING", "approval_type": "manual", "amount": refund_amount, "next_step": "awaiting_manager_approval"}

# Test cases
print("--- Test Case 1: Small Refund (≤ ₹1000) ---")
result1 = process_refund_with_guardrail("C001", "1001", 500)
print(f"Result: {result1}\n")

print("--- Test Case 2: Large Refund (> ₹1000) ---")
result2 = process_refund_with_guardrail("C002", "1002", 1800)
print(f"Result: {result2}")

--- Test Case 1: Small Refund (≤ ₹1000) ---
Processing Refund:
  Customer ID: C001
  Order ID: 1001
  Refund Amount: ₹500

✓ Guardrail Check: Refund ≤ ₹1000
✓ Decision: AUTOMATIC APPROVAL
✓ Action: Processing refund...
Result: {'status': 'APPROVED', 'approval_type': 'automatic', 'amount': 500}

--- Test Case 2: Large Refund (> ₹1000) ---
Processing Refund:
  Customer ID: C002
  Order ID: 1002
  Refund Amount: ₹1800

⚠ Guardrail Check: Refund > ₹1000
⚠ Decision: HUMAN APPROVAL REQUIRED
⚠ Action: Forwarding to manager for review...
Result: {'status': 'PENDING', 'approval_type': 'manual', 'amount': 1800, 'next_step': 'awaiting_manager_approval'}


## Human-in-the-Loop: Adding Approval Steps

For high-impact decisions, we embed human review directly into the agent workflow.

### The Flow with Human Approval

```
Agent calculates refund amount
      ↓
Python guardrail checks: Is refund > ₹1000?
      ↓
   YES → Send to human for approval
      ↓
Human reviews: Approve or Deny
      ↓
   If APPROVED → Execute process_refund()
   If DENIED   → Stop (no refund)
```

### Benefits

1. **Control:** Humans approve high-risk actions
2. **Accountability:** There's a clear audit trail
3. **Compliance:** Meets regulatory requirements
4. **Trust:** Business stakeholders feel confident

### Real-World Implementation

In production systems:
- High-value refunds → Manager approval
- Sensitive data access → Admin approval
- Large transactions → Finance team review

### Key Takeaway

Agents are powerful, but critical decisions should include human judgment.

In [9]:
# Human-in-the-Loop Workflow Simulation

def human_in_the_loop_refund(customer_id, order_id, refund_amount):
    """
    Simulate a refund workflow with human approval step.
    """
    AUTO_REFUND_LIMIT = 1000
    
    print(f"=== Refund Request ===")
    print(f"Customer: {customer_id}")
    print(f"Order: {order_id}")
    print(f"Amount: ₹{refund_amount}\n")
    
    # Step 1: Guardrail check
    if refund_amount <= AUTO_REFUND_LIMIT:
        print(f"✓ Automatic approval (amount ≤ ₹{AUTO_REFUND_LIMIT})")
        print(f"✓ Refund processed automatically.\n")
        return {"status": "PROCESSED", "method": "automatic"}
    else:
        print(f"⚠ Amount exceeds auto-approval limit (₹{refund_amount} > ₹{AUTO_REFUND_LIMIT})")
        print(f"→ Sending to manager for human approval...\n")
        
        # Step 2: Simulate human approval
        # In a real system, this would wait for input
        human_approval = True  # Simulating manager approval
        
        if human_approval:
            print(f"✓ Manager approved the refund.")
            print(f"✓ Refund processed.\n")
            return {"status": "PROCESSED", "method": "manual_approval"}
        else:
            print(f"✗ Manager denied the refund.")
            print(f"✗ Refund request cancelled.\n")
            return {"status": "DENIED", "reason": "Manager rejected"}

# Test both scenarios
print("Scenario 1: Small Refund (Automatic)\n")
result1 = human_in_the_loop_refund("C001", "1001", 750)
print("\n" + "="*40 + "\n")

print("Scenario 2: Large Refund (Requires Approval)\n")
result2 = human_in_the_loop_refund("C002", "1002", 2500)
print(f"Refund Results:")
print(f"  Scenario 1 (₹750): {result1['status']}")
print(f"  Scenario 2 (₹2500): {result2['status']}")

Scenario 1: Small Refund (Automatic)

=== Refund Request ===
Customer: C001
Order: 1001
Amount: ₹750

✓ Automatic approval (amount ≤ ₹1000)
✓ Refund processed automatically.



Scenario 2: Large Refund (Requires Approval)

=== Refund Request ===
Customer: C002
Order: 1002
Amount: ₹2500

⚠ Amount exceeds auto-approval limit (₹2500 > ₹1000)
→ Sending to manager for human approval...

✓ Manager approved the refund.
✓ Refund processed.

Refund Results:
  Scenario 1 (₹750): PROCESSED
  Scenario 2 (₹2500): PROCESSED


## Limitations and Best Practices

Agents are powerful, but they have real limitations. Understanding these is crucial for building reliable systems.

### Common Limitations

| Limitation | Description | Mitigation |
|------------|-------------|------------|
| **Hallucinations** | LLMs can make wrong decisions or invent information | Validate decisions; use guardrails |
| **Incorrect Arguments** | Tool arguments might be wrong or malformed | Validate inputs before execution |
| **Tool Failures** | External tools can fail, timeout, or return errors | Add error handling and retries |
| **Unnecessary Steps** | Agent might call tools that aren't needed | Clear tool descriptions; limit available tools |
| **Infinite Loops** | Agent can get stuck calling the same tool repeatedly | Add iteration limits; monitor loops |
| **High Costs** | Multiple LLM calls can be expensive | Cache results; use cheaper models; limit calls |
| **Latency** | Multi-step agents are slower than direct calls | Accept tradeoff; optimize where possible |

### Best Practices

1. **Use Deterministic Code:** Where possible, avoid agent decisions. Use Python logic instead.
   ```python
   # Good: Deterministic business logic
   if refund_amount > 1000:
       require_approval()
   ```

2. **Write Clear Tool Descriptions:** Help the LLM understand when to use each tool.
   ```python
   # Good: Clear, specific
   "Calculate total order cost including taxes and shipping"
   
   # Bad: Vague
   "Calculate cost"
   ```

3. **Limit Available Tools:** Fewer tools = fewer wrong decisions.
   ```python
   # Expose only the tools the agent needs for this task
   tools = [get_customer, calculate_discount]  # Not all tools
   ```

4. **Validate Tool Inputs:** Always check arguments before execution.
   ```python
   def get_customer(customer_id):
       if not isinstance(customer_id, str) or len(customer_id) == 0:
           raise ValueError("Invalid customer_id")
       # ... proceed
   ```

5. **Add Human Approval:** Critical actions need human judgment.
   ```python
   if is_high_risk_action():
       wait_for_human_approval()
   ```

6. **Monitor and Log:** Track all agent actions for debugging.
   ```python
   log(f"Tool called: {tool_name}, Args: {args}, Result: {result}")
   ```

7. **Set Iteration Limits:** Prevent infinite loops.
   ```python
   max_iterations = 5
   if iteration_count >= max_iterations:
       stop_agent()
   ```

## Summary: From LLM to Agent

Let's recap the complete journey:

### The Core Loop

```
User Input
    ↓
LLM (with tools available)
    ↓
LLM Decision: What should happen next?
    ↓
Tool Call (if needed) OR Final Response
    ↓
Python Executes Tool
    ↓
Tool Result
    ↓
LLM Processes Result
    ↓
Final Response to User
```

### What You've Learned

✅ **LLM vs. Agent:** LLM answers questions; Agent reasons and acts.

✅ **Tool Calling:** LLM decides to call a tool; Application executes it.

✅ **Custom Tools:** Python functions exposed to the LLM for real-world capabilities.

✅ **Agent Loop:** Cycle of decision → action → result → decision.

✅ **Task-Oriented Agents:** Chain multiple steps to accomplish complex goals.

✅ **Guardrails:** Python enforces critical business rules, not the LLM.

✅ **Human-in-the-Loop:** High-impact decisions include human approval.

✅ **Best Practices:** Validation, clear descriptions, limits, logging, deterministic code.

### The Formula

```
LLM + Tools + Decision-Making Logic + Control Flow = Agentic Application
```

---

**Next Steps:**
- Apply these concepts to build real agents
- Experiment with different tool combinations
- Implement guardrails for your domain
- Think about where human approval is necessary

You now have the foundation to build intelligent, reasoning applications! 🚀